# Shin et al. (2025) from dandiset 000248: decoding

The decoding claims of the Neuropixels data (Figs. 3a, 3c, 3d; Extended Data Fig. 7), on
[DANDI:000248](https://dandiarchive.org/dandiset/000248/0.240502.2344) version 0.240502.2344. The
header of `reproduce_shin_2025.ipynb` gives the paper, the text the claims file was written from, the
choice of version and the other claims. Loading and the decoder are in `illusion.py`.

| Claim | Tested here |
|---|---|
| fig3a-npx-v1-decoding | yes |
| fig3c-npx-v1-inference | yes (seeds in `decoding_checks_shin_2025.ipynb`) |
| fig3d-npx-areas-inference | yes (seeds in `decoding_checks_shin_2025.ipynb`) |
| ed7b-npx-xre-inference | yes |
| ed7a-npx-no-area-above-v1 | yes |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import time
import warnings

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy import stats

import illusion as il

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
N_JOBS = 4

started = time.time()
sessions = il.load_all(ccg=False)
print(f"loaded {len(sessions)} sessions of dandiset {il.DANDISET_ID} version {il.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

RS units per area and session, and ICwcfg1 trials of the decoded images.

In [ ]:
rows = []
for s in sessions:
    u, b = s["units"], s["blocks"]["ICwcfg1"]
    n = {c: int(np.sum(b["codes"] == c)) for c in [il.IC1, il.LC1, il.LC2, il.IC2, il.TRE1, il.TRE2, il.XRE1, il.XRE2]}
    rows.append({"subject": s["subject"], **{f"{a}_RS": int(((u.area == a) & u.rs).sum()) for a in il.AREAS},
                 "trials IC1/LC1/LC2/IC2": "/".join(str(n[c]) for c in [il.IC1, il.LC1, il.LC2, il.IC2]),
                 "TRE1/TRE2": f"{n[il.TRE1]}/{n[il.TRE2]}", "XRE1/XRE2": f"{n[il.XRE1]}/{n[il.XRE2]}"})
units_tab = pd.DataFrame(rows)
print(units_tab.to_string(index=False))
allR = np.concatenate([s["blocks"]["ICwcfg1"]["R"].ravel() for s in sessions])
print(f"IC response (spikes/s): min {allR.min():.1f}, median {np.median(allR):.1f}, max {allR.max():.1f}")
print("Session-areas with >= 50 RS units:", {a: int((units_tab[f"{a}_RS"] >= 50).sum()) for a in il.AREAS},
      "(paper 12, 8, 11, 10, 12, 11)")

The counts of RS units and trials are those of the main notebook's data check (400 trials of each
decoded image per session). The clock of spike times and trial table is checked there.

## Decoder

The decoder of the claims file: ICwcfg1, RS units of one area, sessions with at least 50 RS units
in the area; IC response; classes IC1, LC1, LC2, IC2 balanced to the smallest class by its first
trials; stratified tenfold partition; z-scoring by the training trials; linear SVM, one-vs-one.
Reading (a): C = 1; reading (b): C chosen by an inner threefold cross-validation on the training
trials over 10^-3 to 1, the Python counterpart of the code's hyperparameter search. The search
over C above 1, the one-vs-all coding and the kernel scale (equivalent to C for a linear kernel)
is examined in `decoding_checks_shin_2025.ipynb`. TRE trials are decoded by each fold model and
the fractions averaged. The XRE decoders are trained on XRE1 against XRE2 and probed with IC1 and
IC2. Each (session, area) has its own seed. Both readings are run on every area.

In [ ]:
t0 = time.time()
jobs = []
for i, s in enumerate(sessions):
    u = s["units"]
    for j, a in enumerate(il.AREAS):
        m = ((u.area == a) & u.rs).values
        if m.sum() < 50:
            continue
        b = s["blocks"]["ICwcfg1"]
        jobs.append((s["subject"], a, b["R"][:, m], b["codes"], 1000 * i + j))

def run_decoders(subject, a, R, codes, seed):
    b = {"R": R, "codes": codes}
    m = np.ones(R.shape[1], bool)
    out = []
    for rd in ["a", "b"]:
        r = il.decode(b["R"][:, m], b["codes"], [il.IC1, il.LC1, il.LC2, il.IC2], [il.TRE1, il.TRE2], seed=seed, reading=rd)
        score, icv, lcv = il.inference_score(r["probe"])
        x = il.decode(b["R"][:, m], b["codes"], [il.XRE1, il.XRE2], [il.IC1, il.IC2], seed=seed, reading=rd)
        out.append({"subject": subject, "area": a, "n_units": int(m.sum()), "reading": rd,
                    "acc": r["acc"], "tre_ic": icv, "tre_lc": lcv, "inference": score, "n_per_class": r["n_per_class"],
                    "xre_acc": x["acc"], "xre_score": (x["probe"][0, 0] + x["probe"][1, 1]) / 2,
                    "C_tre": r["C_chosen"], "C_xre": x["C_chosen"], "unconverged": r["n_unconverged"] + x["n_unconverged"]})
    return out

dec = pd.DataFrame([r for rs in Parallel(n_jobs=N_JOBS)(delayed(run_decoders)(*j) for j in jobs) for r in rs])
warnings.filterwarnings("ignore")
print(f"{len(jobs)} session-area decoders, both readings, in {time.time() - t0:.0f} s")
print("Sessions per area:", dec[dec.reading == "a"].groupby("area").size().reindex(il.AREAS).to_dict(),
      "(paper 12, 8, 11, 10, 12, 11)")
print("Trials per class:", sorted(dec.n_per_class.unique()))
print("Fits stopped at max_iter without converging:", dec.groupby("reading").unconverged.sum().to_dict(),
      "of", len(jobs) * 2 * 10, "outer fits per reading (reading b: each with 12 inner fits and a refit)")
cb = dec[dec.reading == "b"]
print("Reading (b), C chosen per fold, TRE decoder:", pd.Series(np.concatenate(cb.C_tre.values)).value_counts().sort_index().to_dict(),
      "; XRE decoder:", pd.Series(np.concatenate(cb.C_xre.values)).value_counts().sort_index().to_dict())

## fig3a-npx-v1-decoding

> A linear SVM decodes the four images IC1, LC1, LC2 and IC2 from V1 activity in the Neuropixels
> data with tenfold cross-validation accuracy above the chance of 0.25.

In [ ]:
for rd in ["a", "b"]:
    x = dec[(dec.reading == rd)]
    v = x[x.area == "V1"].acc
    print(f"reading ({rd}) V1: mean accuracy {v.mean():.3f} +/- {v.sem():.3f} (s.e.m.), range {v.min():.3f} to {v.max():.3f}, "
          f"n = {len(v)}, signed-rank against 0.25 p = {il.signrank_p(v - 0.25):.4g}; sessions above 0.25: {(v > 0.25).sum()}")
    if rd == "a":
        print("   other areas:", {a: (round(x[x.area == a].acc.mean(), 3), round(il.signrank_p(x[x.area == a].acc - 0.25), 4))
                                 for a in il.AREAS[1:]})
print(dec[(dec.reading == "a") & (dec.area == "V1")][["subject", "n_units", "acc"]].round(3).to_string(index=False))

**Result: reproduced.** V1 decoding accuracy is 0.697 under reading (a) and 0.719 under reading (b), against the chance of 0.25, with a signed-rank p = 0.00049 under both (12 sessions, every one above 0.25).

## fig3c-npx-v1-inference

> In Neuropixels V1, TRE trials are decoded as the matching IC image more often than as the
> matching LC image (0.51 against 0.22), so IC inference is represented in V1.

In [ ]:
for rd in ["a", "b"]:
    x = dec[(dec.reading == rd) & (dec.area == "V1")]
    print(f"reading ({rd}): P(TRE -> IC) {x.tre_ic.mean():.3f} +/- {x.tre_ic.sem():.3f}, P(TRE -> LC) {x.tre_lc.mean():.3f} +/- "
          f"{x.tre_lc.sem():.3f}; IC > LC in {(x.tre_ic > x.tre_lc).sum()} of {len(x)} sessions; "
          f"signed-rank IC vs LC p = {il.signrank_p(x.tre_ic, x.tre_lc):.4g}")
print("Paper: 0.51 (tolerance 0.051) and 0.22 (0.022), P < 0.05")
print(dec[(dec.area == "V1")].pivot_table(index="subject", columns="reading", values=["tre_ic", "tre_lc"]).round(3).to_string())

**Result: reproduced.** Under reading (b), TRE is decoded as the matching IC with probability 0.518 and as the matching LC with 0.210, both within 0.51 +/- 0.051 and 0.22 +/- 0.022, with IC above LC at signed-rank p = 0.0024. Under reading (a), IC is 0.516 and LC 0.197, just below the lower bound 0.198, with the same p. With two further cross-validation seeds (`decoding_checks_shin_2025.ipynb`) both values are within tolerance under both readings (LC 0.198 to 0.215) and p is at most 0.0068.

## fig3d-npx-areas-inference

> In the Neuropixels data almost every visual area decodes TRE as IC more than as LC, significantly
> in V1, LM, RL, AL and PM and not in AM, with V1, LM and AL the strongest.

In [ ]:
paper_p = {"V1": 0.0132, "LM": 0.0156, "RL": 0.0244, "AL": 0.0039, "PM": 0.0342, "AM": 0.1748}
for rd in ["a", "b"]:
    rows = []
    for a in il.AREAS:
        x = dec[(dec.reading == rd) & (dec.area == a)]
        rows.append({"area": a, "sessions": len(x), "IC": x.tre_ic.mean(), "LC": x.tre_lc.mean(), "inference_mean": x.inference.mean(),
                     "inference_sem": x.inference.sem(), "p": il.signrank_p(x.inference), "paper_p": paper_p[a]})
    t = pd.DataFrame(rows)
    t["sig_IC_above_LC"] = (t.p < 0.05) & (t.inference_mean > 0)
    print(f"reading ({rd})")
    print(t.round(4).to_string(index=False))
    print(f"  {int(t.sig_IC_above_LC.sum())} areas significant")
    print("  significant areas:", t[t.sig_IC_above_LC].area.tolist(),
          "; three largest mean scores:", t.nlargest(3, "inference_mean").area.tolist(),
          "; three smallest p:", t.nsmallest(3, "p").area.tolist())
print("Per-session inference scores, reading (a) and (b):")
print(dec.pivot_table(index="subject", columns=["reading", "area"], values="inference").round(3).to_string())

**Result: partially reproduced, resting on an assumption.** The readings differ. Under reading (a), five areas decode TRE as IC significantly more than as LC (V1, LM, RL, AL and AM; PM p = 0.092), and V1, LM and AL have the three largest mean scores, which meets the criterion; the two further seeds give the same five areas. Under reading (b), four areas are significant (V1, LM, RL and AL; PM p = 0.077, AM p = 0.083), and LM, V1 and AL have the three largest mean scores, a partial reproduction. Reading (b) is adopted, as the paper's code chooses the SVM hyperparameters by a search, which reading (a) leaves out. Under neither reading does the set of significant areas match the paper's, which has PM significant and AM not.

## ed7b-npx-xre-inference

> Decoders trained on the XRE real-bar images classify IC1 and IC2 trials as the XRE image whose
> real bar matches the illusory bar, above chance in all six Neuropixels areas.

XRE inference score (P(IC1 -> XRE1) + P(IC2 -> XRE2))/2, signed-rank against 0.5, the chance of a two-class decoder.

In [ ]:
paper_xp = {"V1": 0.0005, "LM": 0.0078, "RL": 0.0049, "AL": 0.0020, "PM": 0.0005, "AM": 0.0137}
for rd in ["a", "b"]:
    rows = []
    for a in il.AREAS:
        x = dec[(dec.reading == rd) & (dec.area == a)]
        rows.append({"area": a, "sessions": len(x), "xre_acc": x.xre_acc.mean(), "xre_score_mean": x.xre_score.mean(),
                     "xre_score_sem": x.xre_score.sem(), "p_vs_0.5": il.signrank_p(x.xre_score - 0.5), "paper_p": paper_xp[a]})
    t = pd.DataFrame(rows)
    t["sig_above_0.5"] = (t["p_vs_0.5"] < 0.05) & (t.xre_score_mean > 0.5)
    print(f"reading ({rd}): areas significant above 0.5: {int(t['sig_above_0.5'].sum())} of 6")
    print(t.round(4).to_string(index=False))


**Result: reproduced.** The XRE inference score is above 0.5 with signed-rank p < 0.05 in all six areas under both readings, with p from 0.0005 to 0.0137, close to the paper's values (0.0005 to 0.0137).

## ed7a-npx-no-area-above-v1

> No higher visual area shows a substantially larger inference performance than V1, although
> decoding accuracy and inference scores differ across areas.

Kruskal-Wallis across the six areas of the per-session values, then (a) two-sided signed-rank of
V1 against each HVA over the sessions with both, and (b) Tukey-Kramer comparisons after the
Kruskal-Wallis. A claim of absence: reproduced when no HVA is significantly larger than V1.

In [ ]:
def tukey_vs_v1(vals_by_area):
    # Tukey-Kramer on mean ranks after Kruskal-Wallis (MATLAB multcompare default), HVA minus V1
    allv = np.concatenate([vals_by_area[a] for a in il.AREAS])
    lab = np.concatenate([[a] * len(vals_by_area[a]) for a in il.AREAS])
    r = stats.rankdata(allv)
    N = len(allv)
    _, c = np.unique(allv, return_counts=True)
    var = N * (N + 1) / 12 * (1 - np.sum(c ** 3 - c) / (N ** 3 - N))
    qc = il.tukey_qcrit(6)
    out = {}
    for a in il.AREAS[1:]:
        d = r[lab == a].mean() - r[lab == "V1"].mean()
        se = np.sqrt(var * (1 / np.sum(lab == a) + 1 / np.sum(lab == "V1")) / 2)
        out[a] = (round(d, 1), bool(abs(d) / se > qc))
    return out

for rd in ["a", "b"]:
    x = dec[dec.reading == rd]
    print(f"reading ({rd})")
    for meas, paper in [("acc", 0.0019), ("inference", 0.0249), ("xre_acc", 0.0005), ("xre_score", 0.0022)]:
        vals = {a: x[x.area == a][meas].values for a in il.AREAS}
        kwp = stats.kruskal(*vals.values()).pvalue
        piv = x.pivot(index="subject", columns="area", values=meas)
        pair = {}
        for a in il.AREAS[1:]:
            both = piv[["V1", a]].dropna()
            pair[a] = (round(both[a].mean() - both["V1"].mean(), 3), round(il.signrank_p(both[a], both["V1"]), 4))
        print(f"  {meas}: Kruskal-Wallis p = {kwp:.4f} (paper {paper}); means", {a: round(v.mean(), 3) for a, v in vals.items()})
        print(f"     (a) HVA minus V1 and signed-rank p:", pair)
        print(f"     (b) Tukey-Kramer mean-rank difference HVA minus V1, significant:", tukey_vs_v1(vals))

**Result: reproduced.** No higher visual area has a significantly larger TRE or XRE inference score than V1 under either pairwise test or either decoder reading: every significant difference has the HVA below V1. The Kruskal-Wallis tests across areas are significant, as in the paper (reading a: p = 0.0006, 0.0071, 0.0005 and 0.0077 for accuracy, inference score, XRE accuracy and XRE score; paper 0.0019, 0.0249, 0.0005, 0.0022).

In [ ]:
print(f"Notebook ran in {(time.time() - started) / 60:.1f} min")

## Summary

| Claim | Outcome |
|---|---|
| fig3a-npx-v1-decoding | reproduced |
| fig3c-npx-v1-inference | reproduced (within tolerance under reading b; under reading a with two of three seeds) |
| fig3d-npx-areas-inference | partially reproduced (assumption: decoder reading b; reading a meets the criterion) |
| ed7b-npx-xre-inference | reproduced |
| ed7a-npx-no-area-above-v1 | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: the decoders use scikit-learn's libsvm SVM in place of
MATLAB's fitcecoc. The code's Bayesian hyperparameter search is replaced by a grid search over C
from 10^-3 to 1 with one-vs-one coding (reading b); `decoding_checks_shin_2025.ipynb` shows that C
above 1 never gave a higher inner accuracy in V1 and that the one-vs-all coding changed it by at
most 0.024. Extended Data Fig. 7c pools Neuropixels and 2p areas; its Kruskal-Wallis values are
compared here on the six Neuropixels areas only.